In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
pip install transformers torch sentencepiece streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 25.4 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

In [ ]:
model_name = "humarin/chatgpt_paraphraser_on_T5_base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
print('model loaded')

In [ ]:
def paraphrase_text(text):
    input_text = "paraphrase: " + text + " </s>"

    encoding = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    outputs = model.generate(
    input_ids=encoding["input_ids"],
    attention_mask=encoding["attention_mask"],
    max_length=128,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.7,
    no_repeat_ngram_size=2,
    num_return_sequences=1
)

    paraphrases = [
        tokenizer.decode(o, skip_special_tokens=True)
        for o in outputs
    ]

    paraphrases = [
        p.strip().capitalize()
        for p in paraphrases
        if "?" not in p and len(p.split()) > 3
    ]

    return paraphrases[0] if len(paraphrases) > 0 else "No output"

In [26]:
import ipywidgets as widgets
from IPython.display import display

text_input = widgets.Text(
    placeholder='Type your sentence here...',
    description='Input:',
    layout=widgets.Layout(width='500px')
)

button = widgets.Button(description="Paraphrase")
output = widgets.Output()

def on_button_click(b):
    with output:
        output.clear_output()
        user_text = text_input.value

        if user_text.strip() == "":
            print("Please enter some text!")
        else:
            result = paraphrase_text(user_text)

            print("Paraphrased Output:\n")
            print(result)

button.on_click(on_button_click)

display(text_input, button, output)

Text(value='', description='Input:', layout=Layout(width='500px'), placeholder='Type your sentence here...')

Button(description='Paraphrase', style=ButtonStyle())

Output()